# Image Classification - EMNIST Letters
Midterm Project | Eko Rudiawan Jamzuri


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)
from skimage.feature import hog
from skimage import exposure
import warnings
warnings.filterwarnings('ignore')


## 1. Load Dataset

In [ ]:
# sesuaikan path ke file CSV yang sudah didownload dari kaggle
TRAIN_PATH = "emnist-letters-train.csv"
TEST_PATH  = "emnist-letters-test.csv"

try:
    df_train = pd.read_csv(TRAIN_PATH, header=None)
    df_test  = pd.read_csv(TEST_PATH,  header=None)
    df_all   = pd.concat([df_train, df_test], ignore_index=True)
    print(f"dataset loaded: {df_all.shape[0]} total rows")

except FileNotFoundError:
    # fallback simulasi kalau file belum ada
    print("file tidak ditemukan, pakai data simulasi dulu")
    np.random.seed(42)
    rows = []
    for cls in range(1, 27):
        for _ in range(200):
            rows.append([cls] + list(np.random.randint(0, 256, 784)))
    df_all = pd.DataFrame(rows)
    print(f"simulasi: {len(df_all)} rows")


## 2. Balanced Sampling (100 per kelas)

In [ ]:
N = 100  # samples per kelas

# shuffle dulu biar samplingnya random
df_all = df_all.sample(frac=1, random_state=42).reset_index(drop=True)

frames = []
for label in sorted(df_all.iloc[:, 0].unique()):
    subset = df_all[df_all.iloc[:, 0] == label]
    frames.append(subset.head(N))

df = pd.concat(frames).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"total setelah sampling: {len(df)} ({df.iloc[:,0].nunique()} kelas)")


In [ ]:
# pisahin label dan pixel
y_raw = df.iloc[:, 0].values
X_raw = df.iloc[:, 1:].values

# konversi label angka ke huruf (1=A, 2=B, ...)
label_map = {i: chr(64 + i) for i in range(1, 27)}
y = np.array([label_map[int(l)] for l in y_raw])

print(f"X shape: {X_raw.shape}")
print(f"kelas: {sorted(set(y))}")


### Cek beberapa sample gambar

In [ ]:
fig, axes = plt.subplots(4, 7, figsize=(13, 8))

for idx, letter in enumerate(sorted(set(y))):
    r, c = divmod(idx, 7)
    if r >= 4:
        break
    i = np.where(y == letter)[0][0]
    img = np.fliplr(X_raw[i].reshape(28, 28).T)
    axes[r][c].imshow(img, cmap='gray')
    axes[r][c].set_title(letter, fontsize=9)
    axes[r][c].axis('off')

for idx in range(26, 28):
    r, c = divmod(idx, 7)
    axes[r][c].axis('off')

plt.suptitle("sample gambar tiap kelas", y=1.01)
plt.tight_layout()
plt.savefig("sample_images.png", dpi=120, bbox_inches='tight')
plt.show()


## 3. Train-Test Split (80/20)

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"train: {len(X_train_raw)}")
print(f"test : {len(X_test_raw)}")


## 4. HOG Feature Extraction

Parameter yang dipakai (beda dari default):
- `orientations=12` — nangkap lebih banyak arah gradien dibanding default 9
- `pixels_per_cell=(4,4)` — resolusi lebih halus, default (8,8)
- `cells_per_block=(3,3)` — normalisasi per blok 3x3 cell


In [ ]:
def extract_hog(X_flat):
    feats = []
    for px in X_flat:
        img = np.fliplr(px.reshape(28, 28).astype(np.float32).T) / 255.0
        f = hog(img,
                orientations=12,
                pixels_per_cell=(4, 4),
                cells_per_block=(3, 3),
                block_norm='L2-Hys',
                feature_vector=True)
        feats.append(f)
    return np.array(feats)

X_train_hog = extract_hog(X_train_raw)
X_test_hog  = extract_hog(X_test_raw)

print(f"HOG feature length per gambar: {X_train_hog.shape[1]}")


In [ ]:
# visualisasi HOG untuk satu gambar contoh
img_sample = np.fliplr(X_train_raw[0].reshape(28, 28).astype(np.float32).T) / 255.0
_, hog_img = hog(img_sample,
                 orientations=12,
                 pixels_per_cell=(4, 4),
                 cells_per_block=(3, 3),
                 block_norm='L2-Hys',
                 visualize=True,
                 feature_vector=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7, 3))
ax1.imshow(img_sample, cmap='gray')
ax1.set_title('original')
ax1.axis('off')

ax2.imshow(exposure.rescale_intensity(hog_img, in_range=(0, 10)), cmap='magma')
ax2.set_title('HOG (orientations=12, ppc=4)')
ax2.axis('off')

plt.tight_layout()
plt.savefig("hog_viz.png", dpi=120, bbox_inches='tight')
plt.show()


## 5. SVM + Grid Search

In [ ]:
param_grid = {
    'C'     : [0.1, 1, 10],
    'kernel': ['rbf', 'poly'],
    'gamma' : ['scale', 'auto'],
}

gs = GridSearchCV(
    SVC(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
gs.fit(X_train_hog, y_train)

print(f"best params  : {gs.best_params_}")
print(f"best CV score: {gs.best_score_:.4f}")


In [ ]:
# plot hasil grid search
results = pd.DataFrame(gs.cv_results_)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle('Grid Search Results (5-fold CV)')

for ax, kernel in zip(axes, ['rbf', 'poly']):
    sub = results[results['param_kernel'] == kernel].sort_values('param_C')
    for gamma in ['scale', 'auto']:
        g = sub[sub['param_gamma'] == gamma]
        ax.plot(g['param_C'], g['mean_test_score'], marker='o', label=f'gamma={gamma}')
    ax.set_title(f'kernel={kernel}')
    ax.set_xlabel('C')
    ax.set_ylabel('accuracy')
    ax.set_xscale('log')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("gridsearch.png", dpi=120, bbox_inches='tight')
plt.show()


## 6. Evaluasi

In [ ]:
best_model = gs.best_estimator_

y_train_pred = best_model.predict(X_train_hog)
y_test_pred  = best_model.predict(X_test_hog)


In [ ]:
def print_metrics(y_true, y_pred, label):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted')
    rec  = recall_score(y_true, y_pred, average='weighted')
    f1   = f1_score(y_true, y_pred, average='weighted')
    print(f"--- {label} ---")
    print(f"accuracy : {acc:.4f}")
    print(f"precision: {prec:.4f}")
    print(f"recall   : {rec:.4f}")
    print(f"f1-score : {f1:.4f}")
    print()
    return acc, prec, rec, f1

train_scores = print_metrics(y_train, y_train_pred, "Training Set")
test_scores  = print_metrics(y_test,  y_test_pred,  "Test Set")


In [ ]:
print(classification_report(y_test, y_test_pred))


In [ ]:
# confusion matrix
cm = confusion_matrix(y_test, y_test_pred, labels=sorted(set(y_test)))
classes = sorted(set(y_test))

fig, ax = plt.subplots(figsize=(13, 11))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax.set_xticks(range(len(classes)))
ax.set_yticks(range(len(classes)))
ax.set_xticklabels(classes)
ax.set_yticklabels(classes)

thresh = cm.max() / 2
for i in range(len(classes)):
    for j in range(len(classes)):
        ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=7,
                color='white' if cm[i, j] > thresh else 'black')

ax.set_xlabel('predicted')
ax.set_ylabel('actual')
ax.set_title('confusion matrix - test set')
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# f1 score per kelas
report = classification_report(y_test, y_test_pred, output_dict=True)
kelas = sorted([k for k in report if k not in ['accuracy', 'macro avg', 'weighted avg']])
f1s   = [report[k]['f1-score'] for k in kelas]
warna = ['#2ecc71' if v >= 0.7 else '#e67e22' if v >= 0.5 else '#e74c3c' for v in f1s]

fig, ax = plt.subplots(figsize=(13, 4))
bars = ax.bar(kelas, f1s, color=warna, edgecolor='none')
ax.axhline(np.mean(f1s), color='steelblue', linestyle='--', lw=1.5,
           label=f'rata-rata = {np.mean(f1s):.2f}')
ax.set_ylim(0, 1.1)
ax.set_xlabel('kelas')
ax.set_ylabel('f1-score')
ax.set_title('f1-score per kelas (test set)')
ax.legend()
ax.grid(axis='y', alpha=0.3)

for bar, val in zip(bars, f1s):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.2f}', ha='center', fontsize=7.5)

plt.tight_layout()
plt.savefig("f1_per_class.png", dpi=120, bbox_inches='tight')
plt.show()
